# K-Fold and Stratified K-Fold Cross-Validation Lab

Explore the mechanics of K-Fold selection, how dataset size interacts with fold variance, and why Stratified K-Fold is statistically mandatory for imbalanced classification tasks.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Choosing K: Data Size and Score Variance

Observe how estimation standard deviation decreases as sample size $N$ and fold count $K$ grow.

In [ ]:
dataset_sizes = [50, 100, 500, 1000]
K_values = [3, 5, 10, 20]

print(f"{'Size':<8}", end="")
for K in K_values:
    print(f"{'K=' + str(K):<10}", end="")
print()
print("-" * 48)

for n in dataset_sizes:
    X, y = make_classification(n_samples=n, n_features=10, n_classes=2, random_state=42)
    m = LogisticRegression(random_state=42, max_iter=1000)
    print(f"{n:<8}", end="")
    for K in K_values:
        if K > n:
            print(f"{'N/A':<10}", end="")
            continue
        scores = cross_val_score(m, X, y, cv=KFold(n_splits=K, shuffle=True, random_state=42))
        print(f"{np.std(scores):<10.4f}", end="")
    print()

## 2. Standard K-Fold vs. Stratified K-Fold on Imbalanced Data

Simulate severe class imbalance (95% Class 0 vs 5% Class 1) across 100 samples to demonstrate how random splitting creates catastrophic empty minority folds.

In [ ]:
# Generate 95:5 imbalanced data
X_imb, y_imb = make_classification(n_samples=100, n_features=5, n_classes=2, weights=[0.95, 0.05], random_state=42)

print("--- 1. Random K-Fold (Unstratified) ---")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (_, test_idx) in enumerate(kf.split(X_imb)):
    counts = np.bincount(y_imb[test_idx], minlength=2)
    print(f"Fold {fold_idx+1}: Class 0 = {counts[0]:<2} | Class 1 = {counts[1]:<2} -> Ratio {counts[0] / (counts[1] + 1e-6):.1f}:1")

print("\n--- 2. Stratified K-Fold ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (_, test_idx) in enumerate(skf.split(X_imb, y_imb)):
    counts = np.bincount(y_imb[test_idx], minlength=2)
    print(f"Fold {fold_idx+1}: Class 0 = {counts[0]:<2} | Class 1 = {counts[1]:<2} -> Ratio {counts[0] / (counts[1] + 1e-6):.1f}:1")

## 3. Step-by-Step Stratified CV Loop

Manually execute each step of the CV loop to inspect individual fold indices, training fits, and test evaluations.

In [ ]:
X_small = np.random.randn(20, 2)
y_small = np.array([0]*10 + [1]*10)

skf_small = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(random_state=42, max_iter=1000)
fold_scores = []

print(f"{'Fold':<6} {'Train Rows':<12} {'Test Rows':<12} {'Accuracy':<10}")
print("-" * 42)
for i, (train_idx, test_idx) in enumerate(skf_small.split(X_small, y_small)):
    model.fit(X_small[train_idx], y_small[train_idx])
    acc = model.score(X_small[test_idx], y_small[test_idx])
    fold_scores.append(acc)
    print(f"{i+1:<6} {len(train_idx):<12} {len(test_idx):<12} {acc:<10.1%}")

print(f"\nCross-Validated Score: {np.mean(fold_scores):.1%} ± {np.std(fold_scores):.3f}")